In [1]:
%pip install scikit-learn
import re
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score

df = pd.read_csv("../data/clean_dataset.csv")

train, test = train_test_split(df, test_size=0.2, stratify=df["label"], random_state=42)
train.to_csv("../data/train.csv", index=False)
test.to_csv("../data/test.csv", index=False)
print(train.shape, test.shape)
print(test["label"].value_counts())

Note: you may need to restart the kernel to use updated packages.
(16202, 3) (4051, 3)
label
safe            2359
credential       540
email            500
phone            500
url_internal     152
Name: count, dtype: int64


In [2]:
RULES = [
    ("credential", [
        r"(?i)\b(password|passwd|pwd|pass|secret[_ ]?key|api[_-]?key|token|private[_-]?key)\b\s*[:=]?\s*['\"]?\S{4,}",
        r"AKIA[0-9A-Z]{16}",
        r"ghp_[A-Za-z0-9]{20,}",
        r"sk-[A-Za-z0-9]{16,}",
        r"(?i)bearer\s+[A-Za-z0-9._-]{15,}",
    ]),
    ("url_internal", [
        r"(?i)https?://(localhost|127\.\d+\.\d+\.\d+|10\.\d+\.\d+\.\d+|192\.168\.\d+\.\d+|172\.(1[6-9]|2\d|3[01])\.\d+\.\d+)",
        r"(?i)https?://[\w.-]+\.(local|corp|internal|lan)\b",
    ]),
    ("email", [
        r"[\w.+-]+@[\w-]+\.[\w.-]+",
    ]),
    ("phone", [
        r"(\+?\d[\d\s().-]{7,}\d)",
    ]),
]
COMPILED = [(lab, [re.compile(p) for p in pats]) for lab, pats in RULES]

def regex_predict(text):
    for label, patterns in COMPILED:
        if any(p.search(text) for p in patterns):
            return label
    return "safe"

In [3]:
test = pd.read_csv("../data/test.csv")
test["pred"] = test["text"].apply(regex_predict)

labels = ["safe", "email", "phone", "url_internal", "credential"]
print(classification_report(test["label"], test["pred"], labels=labels, digits=3, zero_division=0))
print("Macro F1:", round(f1_score(test["label"], test["pred"], average="macro", labels=labels), 3))
print(pd.DataFrame(confusion_matrix(test["label"], test["pred"], labels=labels),
                   index=labels, columns=labels))

              precision    recall  f1-score   support

        safe      0.917     0.938     0.927      2359
       email      1.000     1.000     1.000       500
       phone      0.764     0.998     0.866       500
url_internal      1.000     1.000     1.000       152
  credential      0.976     0.604     0.746       540

    accuracy                          0.911      4051
   macro avg      0.931     0.908     0.908      4051
weighted avg      0.919     0.911     0.907      4051

Macro F1: 0.908
              safe  email  phone  url_internal  credential
safe          2212      0    140             0           7
email            0    500      0             0           0
phone            0      0    499             0           1
url_internal     0      0      0           152           0
credential     200      0     14             0         326


In [4]:
test = pd.read_csv("../data/test.csv")
test["pred"] = test["text"].apply(regex_predict)

labels = ["safe", "email", "phone", "url_internal", "credential"]
print(classification_report(test["label"], test["pred"], labels=labels, digits=3, zero_division=0))
print("Macro F1:", round(f1_score(test["label"], test["pred"], average="macro", labels=labels), 3))
print(pd.DataFrame(confusion_matrix(test["label"], test["pred"], labels=labels),
                   index=labels, columns=labels))

              precision    recall  f1-score   support

        safe      0.917     0.938     0.927      2359
       email      1.000     1.000     1.000       500
       phone      0.764     0.998     0.866       500
url_internal      1.000     1.000     1.000       152
  credential      0.976     0.604     0.746       540

    accuracy                          0.911      4051
   macro avg      0.931     0.908     0.908      4051
weighted avg      0.919     0.911     0.907      4051

Macro F1: 0.908
              safe  email  phone  url_internal  credential
safe          2212      0    140             0           7
email            0    500      0             0           0
phone            0      0    499             0           1
url_internal     0      0      0           152           0
credential     200      0     14             0         326


In [5]:
# Results per source (shows the source-style effect)
for src, g in test.groupby("source"):
    acc = (g["label"] == g["pred"]).mean()
    print(f"{src:18s} accuracy = {acc:.3f}  (n={len(g)})")

# Sample mistakes
errors = test[test["label"] != test["pred"]]
print(len(errors), "errors")
errors[["text", "label", "pred"]].sample(15, random_state=1)

ag_news            accuracy = 0.993  (n=286)
ai4privacy         accuracy = 0.796  (n=928)
faker_synthetic    accuracy = 0.980  (n=1614)
hard_negatives     accuracy = 0.784  (n=638)
rotten_tomatoes    accuracy = 1.000  (n=576)
ui_code            accuracy = 1.000  (n=9)
362 errors


,text,label,pred
2917,with Adella6 and 3cSG2sDgpc8s.,credential,safe
3386,Essex and passwords CfHZkYR9k01n could leave y...,credential,safe
1121,with Craig.Sawayn & U8oR2fbwIRFH.,credential,safe
1526,"as your last password, '0qAyaO0ofsTZ'.",credential,safe
1995,Invoice #: REQ-45-149588,safe,phone
1881,"Kathleen, I3Q9rVKrYtgz is the password for your",credential,safe
66,DB_PASS = 'WN%1bK8E%o#hQ(Qr',credential,safe
4010,using Quentin32 and qqt7gOduwJFJ.,credential,safe
3856,TXN-57-844162,safe,phone
2410,Date(s) of Service: 19 Oct 1991 to 2023-09-17,safe,phone


In [6]:
import os
os.makedirs("../report", exist_ok=True)
rep = pd.DataFrame(classification_report(test["label"], test["pred"], labels=labels,
                                         output_dict=True, zero_division=0)).T
rep.to_csv("../report/regex_baseline_results.csv")
rep.round(3)

,precision,recall,f1-score,support
safe,0.917,0.938,0.927,2359.000
email,1.000,1.000,1.000,500.000
phone,0.764,0.998,0.866,500.000
url_internal,1.000,1.000,1.000,152.000
credential,0.976,0.604,0.746,540.000
accuracy,0.911,0.911,0.911,0.911
macro avg,0.931,0.908,0.908,4051.000
weighted avg,0.919,0.911,0.907,4051.000


# 05 - Train/Test Split and Regex Baseline

## Purpose
Create one fixed train/test split for the whole team, and build a rule-based (regex)
detector. This is the benchmark that our ML models must beat, and it answers the
question "why use ML instead of regex?".

## Input / Output
- **Input:** `data/clean_dataset.csv`
- **Output:** `data/train.csv` (80%), `data/test.csv` (20%),
  `report/regex_baseline_results.csv`

## What we did
1. Split the data 80/20 with `stratify=label` and `random_state=42`, so every class keeps
   the same proportion and every teammate gets the same split.
2. Wrote regex rules for four classes: credentials (keywords such as password and
   api_key, AKIA keys, ghp_ and sk- tokens, bearer tokens), internal URLs (private IPs,
   localhost, .local/.corp), emails and phones. The first matching rule wins, and
   credentials are checked first. Anything with no match is labeled safe.
3. Ran the rules on the test set only.
4. Measured precision, recall, F1, macro F1 and a confusion matrix.
5. Checked accuracy per source and printed sample mistakes.
6. Saved the scores to the report folder.

## Results (fill in from your output)
- Macro F1: ___
- Best class: ___ (F1 ___)
- Worst class: ___ (F1 ___)
- Accuracy per source: ag_news ___, ai4privacy ___, faker_synthetic ___,
  rotten_tomatoes ___

## Key concepts learned
- **Stratified split** keeps class proportions the same in train and test.
- **Test set is only for final evaluation.** It is never used for training or tuning.
- **Precision** = of the items flagged, how many were really sensitive.
  **Recall** = of the real sensitive items, how many were caught.
  **Macro F1** averages F1 over classes, so small classes count equally.
- A **baseline** gives a fair reference point to prove the ML model adds value.

## Where regex fails (use as the answer to "why ML?")
- Passwords without a keyword (e.g. just `Xk9#mP2q`) are missed.
- Phones in unusual formats are missed, and long safe numbers are flagged by mistake.
- Safe text containing words like "token" or "pass" triggers false alarms.
- Rules only match fixed patterns and cannot use context. ML learns patterns from data.

## Brief requirement covered
Appropriate evaluation metrics, comparison of solution strategies (CA 2).